# Week 6 Homework
### Use any model and compare to openai on 3 tasks of your choice.
### Give your opinions on which performs better and why

### I have used 2 models 
    1. chatgpt-3.5 turbo chatgpt
    2. Mistral 7B instruct

### I tried 3 different tasks - 
1. Summarization
2. Text Generation - story writing
3. Question Answering - Arithmatic operations

##### NOTE: For Mistral 7B instruct, i have ran it on gpu machine in my office, environment, hence copy pasted the code and output in this notebook.

### OpenAI

In [ ]:
# Install dependencies for local Python 3.10 and Colab


In [ ]:
import os
import openai
from openai import OpenAI

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError(
        "Set OPENAI_API_KEY in your environment before running this notebook. "
        "Do not paste secrets into notebook cells."
    )

openai.api_key = OPENAI_API_KEY
client = OpenAI(api_key=OPENAI_API_KEY)
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-3.5-turbo")


In [ ]:
# OpenAI client is initialized in the previous cell.


### Task 1 - Summarization

#### OpenAI

In [ ]:
#text summarizer
response = client.chat.completions.create(
  model=OPENAI_MODEL,
  messages=[
    {"role": "system", "content": "Summarize the text into a headline"},
    {"role": "user", "content": """
          Spain were crowned European champions for a record fourth time as they brought to an end a rollercoaster month of football when
          they defeated England 2-1 in the Euro 2024 final. Hosts Germany were among the pretournament favourites alongside Qatar 2022 World
          Cup finalists France while big names, including Portugal’s Cristiano Ronaldo, were making their final appearance at the competition.
        """}
  ]
)

summary = response.choices[0].message.content
print(summary)


#### Mistral 7B

In [ ]:
import os

RUN_LOCAL_MODEL = os.getenv("RUN_LOCAL_MODEL", "false").lower() == "true"
LOCAL_MODEL_NAME = os.getenv("LOCAL_MODEL_NAME", "mistralai/Mistral-7B-Instruct-v0.1")
model = None
tokenizer = None

def select_device():
    try:
        import torch
        if torch.cuda.is_available():
            return "cuda"
        if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
            return "mps"
    except Exception:
        pass
    return "cpu"

device = select_device()
print(f"Local model device: {device}")


In [ ]:
if RUN_LOCAL_MODEL:
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(LOCAL_MODEL_NAME)
    print(f"Loaded local model: {LOCAL_MODEL_NAME}")
else:
    print(
        "Skipping local model load. Set RUN_LOCAL_MODEL=true to download and run "
        f"{LOCAL_MODEL_NAME}."
    )


In [ ]:
text = "<s>[INST] Summarize the text given in single quote into a headline: [/INST] 'Spain were crowned European champions for a record fourth time as they brought to an end a rollercoaster month of football when they defeated England 2-1 in the Euro 2024 final. Hosts Germany were among the pretournament favourites alongside Qatar 2022 World Cup finalists France while big names, including Portugal’s Cristiano Ronaldo were making their final appearance at the competition.' </s>"


In [ ]:
def call_mistral(text):
    if model is None or tokenizer is None:
        print("Local model not loaded. Set RUN_LOCAL_MODEL=true to run this cell.")
        return None

    encodeds = tokenizer(text, return_tensors="pt", add_special_tokens=False)
    model_inputs = encodeds.to(device)
    model.to(device)
    generated_ids = model.generate(**model_inputs, max_new_tokens=500, do_sample=True)
    decoded = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    print(decoded[0])
    return decoded[0]


In [ ]:
call_mistral(text)


In [ ]:
"""
<s>[INST] Summarize the text given in single quote into a headline: [/INST] 'Spain were crowned European champions for a record fourth time as they brought to an end a rollercoaster month of football when they defeated England 2-1 in the Euro 2024 final. Hosts Germany were among the pretournament favourites alongside Qatar 2022 World Cup finalists France while big names, including Portugal’s Cristiano Ronaldo were making their final appearance at the competition.' </s>

Headline: Spain win Euro 2024 Final, Cristiano Ronaldo's last tournament.</s>
"""


### Task 2 - Generation

#### OpenAI

In [ ]:
response = client.chat.completions.create(
  model=OPENAI_MODEL,
  messages=[
    {"role": "system", "content": "You are a story writer for a movies."},
    {"role": "user", "content": """
          A guy started his career with a job of 3000 USD per year. With a hard work, and upskilling he saved a lot of money. After 3 years he bought a 3 bedroom apartment. After 7 years he started earning USD 130000 per year, with a networth of USD 300000 in real estate, stocks, and debt funds, at the age of 33. His goal is to make his portfolio of 5 million dollars at the age of 40. please ccomplete the story between 33 yo and 40 yo. 
        """}
  ]
)

story = response.choices[0].message.content
print(story)


#### Mistral 7B instruct

In [ ]:
text = "<s>[INST] You are a story writer for a movies. please ccomplete the above story in single quotes, for a guy between 33 yo and 40 yo [/INST] 'A guy started his career with a job of 3000 USD per year. With a hard work, and upskilling he saved a lot of money. After 3 years he bought a 3 bedroom apartment. After 7 years he started earning USD 130000 per year, with a networth of USD 300000 in real estate, stocks, and debt funds, at the age of 33. His goal is to make his portfolio of 5 million dollars at the age of 40. please ccomplete the story between 33 yo and 40 yo.' </s>"


In [ ]:
call_mistral(text)


In [ ]:
"""<s>[INST] You are a story writer for a movies. please ccomplete the above story in single quotes, for a guy between 33 yo and 40 yo [/INST] 'A guy started his career with a job of 3000 USD per year. With a hard work, and upskilling he saved a lot of money. After 3 years he bought a 3 bedroom apartment. After 7 years he started earning USD 130000 per year, with a networth of USD 300000 in real estate, stocks, and debt funds, at the age of 33. His goal is to make his portfolio of 5 million dollars at the age of 40. please ccomplete the story between 33 yo and 40 yo.' </s> ---------------------------------------------------------------------

As the guy approached his 40th year, his portfolio was just under 5 million dollars. However, he had other plans in his mind. He wanted to use some of the money from his portfolio to take up new businesses. He had a long term vision to create a multi-billion dollar empire.

He started a restaurant, a grocery store, and a clothing store. The first one opened in his hometown, as his first business. He also invested in the stock market, and managed to buy out multiple companies. He put his heart and soul into his business, and it paid off.

With that being said, he knew he had to continue growing his businesses. He also started a private club for wealthy individuals, where they could relax, network, and enjoy amenities such as spas and gyms.

His investment strategies in stock, and real estate brought him a high return. He used the proceeds to open restaurants in other countries, and bought out a large portion of the stock market.

His hard work and determination paid off, as he had reached his 40th year, and his portfolio reached 10 million dollars. He continued to grow his empire, and by age 50, he had reached a net worth of 15 million dollars. His journey was a testament to hard work, persistence, and a strong plan. He had successfully achieved his goal of having a portfolio of 5 million dollars at the age of 40, and he had even surpassed it.</s>"""


### Task 3 - Question Answering

#### Openai

In [ ]:
response = client.chat.completions.create(
  model=OPENAI_MODEL,
  messages=[
    {"role": "system", "content": "Answer the following question"},
    {"role": "user", "content": """ Suppose I have a corpus of $100000. If I invest it in a index fund for next 20 years, at a compounded annual rate of 12%, what would be my corpus size after 20 years. """}
  ]
)

ANSWER = response.choices[0].message.content
print(ANSWER)


In [ ]:
text = "<s>[INST] Answer the above question given in single quotes [/INST]  Suppose I have a corpus of $100000. If I invest it in a index fund for next 20 years, at a compounded annual rate of 12%, what would be my corpus size after 20 years. </s>"


In [ ]:
call_mistral(text)


In [ ]:
"""
<s>[INST] Answer the above question given in single quotes [/INST]  Suppose I have a corpus of $100000. If I invest it in a index fund for next 20 years, at a compounded annual rate of 12%, what would be my corpus size after 20 years. </s>

The formula to calculate the future value of an investment with a fixed rate of return compounded annually over a specified time period is:

Future Value = Principal * (1 + Rate of Return)^Number of Years

In this case:
Principal = $100000
Rate of Return = 12% = 0.12
Number of Years = 20

Future Value = $100000 * (1 + 0.12)^20
Future Value = $100000 * (1.12)^20
Future Value = $100000 * 1.25958 (rounded to 6 decimal places)
Future Value = $125958

So, if I invest $100000 in a index fund at a compounded annual rate of 12% for 20 years, my corpus size after 20 years would be approximately $125958.</s>
"""


# Conclusion

"""
I tried 3 different tasks - 
1. Summarization
2. Text Generation - story writing
3. Question Answering - Arithmatic operations

Gpt-3.5 turbo sounds more logical. 
Mistral 7B results is comparable with chatgpt, though it is a very small models compared to chatgpt.

So I think as a open source and a small model, it makes sense to use Mistral 7B for small tasks, and for cost effectiveness.
I have used a 32GB GPU for Mistral 7B inference.
"""